# hybrid v1

- combines heuristic ranking and collaborative filtering
- direct collaboration serving
- same holdout style as the other notebooks


In [1]:
from __future__ import annotations

from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "synthetic"


## Load


In [2]:
collab_table = pd.read_csv(DATA_DIR / "collaboration_training_table.csv")
collabs = pd.read_csv(DATA_DIR / "collaborations.csv")

collab_table["lastEventAt"] = pd.to_datetime(collab_table["lastEventAt"], utc=True)
collab_table["tagsKey"] = collab_table["tagsKey"].apply(literal_eval)
collabs["tagsKey"] = collabs["tagsKey"].apply(literal_eval)

feature_cols = [
    "collaboration_favorite",
    "collaboration_like",
    "submission_favorite",
    "submission_like",
    "submission_vote",
    "totalEventWeight",
]

active_collabs = collabs[collabs["status"].isin(["submission", "voting"])].copy()
active_collabs = active_collabs[["id", "projectId", "name", "status", "tagsKey", "publishedAt"]].rename(columns={"id": "collaborationId"})

collab_table.head()

,userId,projectId,collaborationId,collaboration_favorite,collaboration_like,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt,status,tags,tagsKey,publishedAt
0,user-0001,project-007,collab-007-04,0,0,1,1,0,3.00,2026-04-24 00:22:43+00:00,submission,"['industrial', 'uk-bass']","[industrial, uk-bass]",2026-01-10T03:32:07Z
1,user-0001,project-003,collab-003-01,0,1,1,3,0,5.75,2026-04-03 07:12:35+00:00,submission,"['breakbeat', 'dub', 'jungle']","[breakbeat, dub, jungle]",2026-01-21T00:43:40Z
2,user-0001,project-009,collab-009-02,0,0,1,3,0,5.00,2026-03-28 11:54:51+00:00,voting,['acid'],[acid],2026-01-16T19:47:28Z
3,user-0002,project-007,collab-007-01,0,0,2,2,0,6.00,2026-04-16 05:06:49+00:00,completed,['ambient'],[ambient],2026-02-04T05:59:49Z
4,user-0002,project-012,collab-012-02,0,0,2,2,0,6.00,2026-04-04 14:53:01+00:00,submission,"['breakbeat', 'dub']","[breakbeat, dub]",2026-04-14T19:16:02Z


## Holdout split


In [3]:
eligible_history = collab_table[collab_table["status"].isin(["submission", "voting"])].copy()
eligible_history = eligible_history.sort_values(["userId", "lastEventAt"])

holdout = eligible_history.groupby("userId").tail(1).copy()
train_history = eligible_history.merge(
    holdout[["userId", "collaborationId"]].assign(_holdout=1),
    on=["userId", "collaborationId"],
    how="left",
)
train_history = train_history[train_history["_holdout"].isna()].drop(columns="_holdout")

valid_users = train_history["userId"].value_counts()
valid_users = set(valid_users[valid_users > 0].index) & set(holdout["userId"])
train_history = train_history[train_history["userId"].isin(valid_users)].copy()
holdout = holdout[holdout["userId"].isin(valid_users)].copy()

len(valid_users), train_history.shape, holdout.shape

(239, (1434, 14), (239, 14))

## Heuristic parts


In [4]:
train_popularity = (
    train_history
    .groupby(["collaborationId", "projectId"], as_index=False)[feature_cols]
    .sum()
)
train_popularity["popularityScore"] = (
    1.75 * train_popularity["collaboration_favorite"]
    + 0.75 * train_popularity["collaboration_like"]
    + 2.0 * train_popularity["submission_favorite"]
    + 1.0 * train_popularity["submission_like"]
    + 3.0 * train_popularity["submission_vote"]
)
train_popularity["popularityScoreNorm"] = train_popularity["popularityScore"] / (train_popularity["popularityScore"].max() or 1)
train_popularity = train_popularity[["collaborationId", "projectId", "popularityScoreNorm"]]

tag_rows = []
for row in train_history[["userId", "tagsKey", "totalEventWeight"]].itertuples(index=False):
    tags = row.tagsKey or []
    if not tags:
        continue
    per_tag_weight = row.totalEventWeight / len(tags)
    for tag in tags:
        tag_rows.append({"userId": row.userId, "tag": tag, "weight": per_tag_weight})

train_user_tag_affinity = pd.DataFrame(tag_rows).groupby(["userId", "tag"], as_index=False)["weight"].sum()

train_user_project_affinity = (
    train_history
    .groupby(["userId", "projectId"], as_index=False)["totalEventWeight"]
    .sum()
    .rename(columns={"totalEventWeight": "projectAffinity"})
)
max_project_affinity = train_user_project_affinity.groupby("userId")["projectAffinity"].transform("max")
train_user_project_affinity["projectAffinityNorm"] = train_user_project_affinity["projectAffinity"] / max_project_affinity


## Collaborative filtering parts


In [5]:
interaction_df = (
    train_history.groupby(["userId", "collaborationId"], as_index=False)["totalEventWeight"]
    .sum()
)
interaction_matrix = interaction_df.pivot(index="userId", columns="collaborationId", values="totalEventWeight").fillna(0.0)

X = interaction_matrix.to_numpy(dtype=float)
item_matrix = X.T
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1.0
item_matrix_norm = item_matrix / norms
similarity = item_matrix_norm @ item_matrix_norm.T
np.fill_diagonal(similarity, 0.0)

item_ids = interaction_matrix.columns.tolist()
sim_df = pd.DataFrame(similarity, index=item_ids, columns=item_ids)
sim_df.iloc[:5, :5]

,collab-001-01,collab-001-02,collab-002-01,collab-002-02,collab-002-03
collab-001-01,0.000000,0.141247,0.241948,0.004985,0.077898
collab-001-02,0.141247,0.000000,0.104791,0.260399,0.084873
collab-002-01,0.241948,0.104791,0.000000,0.000000,0.101494
collab-002-02,0.004985,0.260399,0.000000,0.000000,0.030545
collab-002-03,0.077898,0.084873,0.101494,0.030545,0.000000


## Candidate scoring


In [6]:
seen_train = train_history[["userId", "collaborationId"]].drop_duplicates().assign(seen=1)
eval_users = pd.DataFrame({"userId": sorted(valid_users)})

candidates = eval_users.assign(_k=1).merge(active_collabs.assign(_k=1), on="_k").drop(columns="_k")
candidates = candidates.merge(seen_train, on=["userId", "collaborationId"], how="left")
candidates = candidates[candidates["seen"].isna()].drop(columns="seen")

candidates = candidates.merge(train_popularity, on=["collaborationId", "projectId"], how="left")
candidates["popularityScoreNorm"] = candidates["popularityScoreNorm"].fillna(0)

candidates = candidates.merge(
    train_user_project_affinity[["userId", "projectId", "projectAffinityNorm"]],
    on=["userId", "projectId"],
    how="left",
)
candidates["projectAffinityNorm"] = candidates["projectAffinityNorm"].fillna(0)

train_tag_pref = train_user_tag_affinity.groupby("userId").apply(lambda x: dict(zip(x["tag"], x["weight"]))).to_dict()

def tag_score(user_id, tags):
    pref = train_tag_pref.get(user_id, {})
    if not tags:
        return 0.0
    return sum(pref.get(tag, 0.0) for tag in tags) / len(tags)

candidates["tagScoreRaw"] = [tag_score(u, tags) for u, tags in zip(candidates["userId"], candidates["tagsKey"])]
max_tag = candidates.groupby("userId")["tagScoreRaw"].transform("max").replace(0, 1)
candidates["tagScoreNorm"] = candidates["tagScoreRaw"] / max_tag
candidates["heuristicScore"] = (
    0.50 * candidates["tagScoreNorm"]
    + 0.30 * candidates["projectAffinityNorm"]
    + 0.20 * candidates["popularityScoreNorm"]
)

pop_map = dict(zip(train_popularity["collaborationId"], train_popularity["popularityScoreNorm"]))
cf_rows = []
active_ids = candidates["collaborationId"].unique().tolist()
for user_id, row in interaction_matrix.iterrows():
    seen_weights = row[row > 0]
    seen_ids = set(seen_weights.index)
    user_scores = {}
    if len(seen_weights) > 0:
        seen_vector = seen_weights.to_numpy(dtype=float)
        sims = sim_df[seen_weights.index].mul(seen_vector, axis=1).sum(axis=1)
        user_scores = sims.to_dict()
    for collab_id in active_ids:
        if collab_id in seen_ids:
            continue
        cf_rows.append({
            "userId": user_id,
            "collaborationId": collab_id,
            "cfScoreRaw": float(user_scores.get(collab_id, 0.0)),
            "cfPopularityNorm": float(pop_map.get(collab_id, 0.0)),
        })

cf_scores = pd.DataFrame(cf_rows)
max_cf = cf_scores.groupby("userId")["cfScoreRaw"].transform("max").replace(0, 1)
cf_scores["cfScoreNorm"] = cf_scores["cfScoreRaw"] / max_cf
cf_scores["cfScore"] = 0.85 * cf_scores["cfScoreNorm"] + 0.15 * cf_scores["cfPopularityNorm"]

candidates = candidates.merge(cf_scores[["userId", "collaborationId", "cfScoreNorm", "cfScore"]], on=["userId", "collaborationId"], how="left")
candidates["cfScoreNorm"] = candidates["cfScoreNorm"].fillna(0)
candidates["cfScore"] = candidates["cfScore"].fillna(0)

candidates["hybridScore"] = 0.65 * candidates["heuristicScore"] + 0.35 * candidates["cfScore"]
candidates.head()

,userId,collaborationId,projectId,name,status,tagsKey,publishedAt,popularityScoreNorm,projectAffinityNorm,tagScoreRaw,tagScoreNorm,heuristicScore,cfScoreNorm,cfScore,hybridScore
0,user-0001,collab-001-01,project-001,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z,0.716581,0.0,2.305556,0.461111,0.373872,0.390676,0.439562,0.396863
1,user-0001,collab-001-02,project-001,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z,0.894814,0.0,0.638889,0.127778,0.242852,0.265769,0.360125,0.283897
2,user-0001,collab-002-01,project-002,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z,0.341125,0.0,0.000000,0.000000,0.068225,0.300288,0.306414,0.151591
3,user-0001,collab-002-02,project-002,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z,0.327977,0.0,0.000000,0.000000,0.065595,0.170249,0.193908,0.110505
4,user-0001,collab-002-03,project-002,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z,0.322133,0.0,0.000000,0.000000,0.064427,0.405110,0.392663,0.179309


## Top recommendations


In [7]:
top_recommendations = (
    candidates
    .sort_values(["userId", "hybridScore"], ascending=[True, False])
    .groupby("userId")
    .head(10)
    .copy()
)

top_recommendations["rank"] = top_recommendations.groupby("userId").cumcount() + 1
top_recommendations = top_recommendations[[
    "userId",
    "rank",
    "projectId",
    "collaborationId",
    "name",
    "status",
    "tagsKey",
    "tagScoreNorm",
    "projectAffinityNorm",
    "popularityScoreNorm",
    "heuristicScore",
    "cfScoreNorm",
    "cfScore",
    "hybridScore",
]]
top_recommendations.head(20)

,userId,rank,projectId,collaborationId,name,status,tagsKey,tagScoreNorm,projectAffinityNorm,popularityScoreNorm,heuristicScore,cfScoreNorm,cfScore,hybridScore
24,user-0001,1,project-010,collab-010-03,Collaboration 10-3,voting,[acid],1.000000,0.000000,0.637692,0.627538,1.000000,0.945654,0.738879
6,user-0001,2,project-003,collab-003-03,Collaboration 3-3,submission,"[cinematic, lofi, minimal]",0.000000,1.000000,0.769175,0.453835,0.550303,0.583134,0.499090
31,user-0001,3,project-012,collab-012-04,Collaboration 12-4,submission,"[acid, breakbeat, trance]",0.461111,0.000000,0.804967,0.391549,0.637668,0.662763,0.486474
13,user-0001,4,project-006,collab-006-02,Collaboration 6-2,submission,"[acid, breakbeat, trance]",0.461111,0.000000,0.695398,0.369635,0.487011,0.518269,0.421657
5,user-0001,5,project-003,collab-003-02,Collaboration 3-2,submission,[cinematic],0.000000,1.000000,0.371804,0.374361,0.522957,0.500284,0.418434
30,user-0001,6,project-012,collab-012-02,Collaboration 12-2,submission,"[breakbeat, dub]",0.383333,0.000000,0.412710,0.274209,0.696498,0.653930,0.407111
20,user-0001,7,project-009,collab-009-03,Collaboration 9-3,submission,[lofi],0.000000,0.869565,0.623083,0.385486,0.405144,0.437835,0.403808
0,user-0001,8,project-001,collab-001-01,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",0.461111,0.000000,0.716581,0.373872,0.390676,0.439562,0.396863
22,user-0001,9,project-009,collab-009-05,Collaboration 9-5,voting,[idm],0.000000,0.869565,0.453616,0.351593,0.318162,0.338480,0.347003
10,user-0001,10,project-005,collab-005-01,Collaboration 5-1,submission,"[breakbeat, synthwave]",0.191667,0.000000,0.513514,0.198536,0.574024,0.564948,0.326780


## Sanity evaluation


In [8]:
history_tags = train_history.groupby("userId")["tagsKey"].apply(lambda rows: set(t for arr in rows for t in arr)).to_dict()
history_projects = train_history.groupby("userId")["projectId"].apply(lambda s: set(s)).to_dict()

sanity_eval = top_recommendations.copy()
sanity_eval["tagOverlap"] = [len(set(tags) & history_tags.get(uid, set())) for uid, tags in zip(sanity_eval["userId"], sanity_eval["tagsKey"])]
sanity_eval["sameSeenProject"] = [pid in history_projects.get(uid, set()) for uid, pid in zip(sanity_eval["userId"], sanity_eval["projectId"])]

sanity_summary = {
    "rows": int(len(sanity_eval)),
    "users": int(sanity_eval["userId"].nunique()),
    "unique_recommended_collabs": int(sanity_eval["collaborationId"].nunique()),
    "catalog_coverage_pct": round(100 * sanity_eval["collaborationId"].nunique() / len(collabs), 2),
    "active_only": bool(sanity_eval["status"].isin(["submission", "voting"]).all()),
    "top10_tag_overlap_pct": round(100 * (sanity_eval["tagOverlap"] > 0).mean(), 2),
    "top10_same_project_seen_pct": round(100 * sanity_eval["sameSeenProject"].mean(), 2),
    "top1_tag_overlap_pct": round(100 * (sanity_eval[sanity_eval["rank"] == 1]["tagOverlap"] > 0).mean(), 2),
    "top1_same_project_seen_pct": round(100 * sanity_eval[sanity_eval["rank"] == 1]["sameSeenProject"].mean(), 2),
}

sanity_summary

{'rows': 2390,
 'users': 239,
 'unique_recommended_collabs': 38,
 'catalog_coverage_pct': 80.85,
 'active_only': True,
 'top10_tag_overlap_pct': np.float64(91.3),
 'top10_same_project_seen_pct': np.float64(48.74),
 'top1_tag_overlap_pct': np.float64(100.0),
 'top1_same_project_seen_pct': np.float64(61.51)}

In [9]:
top_recommendations[top_recommendations["rank"] == 1]["collaborationId"].value_counts().head(10)

collaborationId
collab-009-03    40
collab-012-04    29
collab-004-01    23
collab-003-03    15
collab-010-03    13
collab-011-02    13
collab-009-02    12
collab-008-01    10
collab-011-04     9
collab-013-02     9
Name: count, dtype: int64

## Holdout evaluation


In [10]:
holdout_eval = holdout[["userId", "collaborationId"]].rename(columns={"collaborationId": "heldOutCollaborationId"})
hits = top_recommendations.merge(holdout_eval, on="userId", how="inner")
hits["isHit"] = hits["collaborationId"] == hits["heldOutCollaborationId"]

user_hits = hits.groupby("userId").agg(
    hitAt10=("isHit", "max"),
    reciprocalRank=("rank", lambda s: 1 / s[hits.loc[s.index, "isHit"]].iloc[0] if hits.loc[s.index, "isHit"].any() else 0),
).reset_index()

holdout_summary = {
    "users_evaluated": int(len(user_hits)),
    "hit_rate_at_10": round(float(user_hits["hitAt10"].mean()), 4),
    "mrr_at_10": round(float(user_hits["reciprocalRank"].mean()), 4),
}

holdout_summary

{'users_evaluated': 239, 'hit_rate_at_10': 0.3598, 'mrr_at_10': 0.1189}